In [ ]:
# Throwaway diagnostic: notebook 1's prepare_dataset.py has sat with zero new
# output for 45+ minutes while scanning ~118K small COCO label .txt files on
# /kaggle/tmp, despite a local benchmark on 120K synthetic files completing in
# under 3 seconds. This isolates whether /kaggle/tmp itself is pathologically
# slow for many-small-file glob+read workloads, independent of any logic bug
# in prepare_dataset.py. No datasets attached - pure filesystem benchmark.
import time
from pathlib import Path

d = Path('/kaggle/tmp/synthtest/labels')
d.mkdir(parents=True, exist_ok=True)
N = 120000

t0 = time.time()
for i in range(N):
    (d / f'{i:08d}.txt').write_text('0 0.5 0.5 0.1 0.1\n3 0.2 0.2 0.05 0.05\n')
    if i and i % 20000 == 0:
        print(f'  wrote {i}, {time.time()-t0:.1f}s', flush=True)
print(f'write done: {time.time()-t0:.1f}s', flush=True)

t0 = time.time()
files = sorted(d.glob('*.txt'))
print(f'glob+sort: {time.time()-t0:.1f}s, {len(files)} files', flush=True)

t0 = time.time()
id_map = {0: 0, 3: 1}
kept_total = 0
for i, f in enumerate(files):
    for line in f.read_text().splitlines():
        parts = line.split()
        if int(parts[0]) in id_map:
            kept_total += 1
    if i and i % 20000 == 0:
        print(f'  scanned {i}, {time.time()-t0:.1f}s', flush=True)
print(f'scan done: {time.time()-t0:.1f}s, kept={kept_total}', flush=True)
